In [6]:
import json
import os
import pandas as pd

# =========================
# CONFIG
# =========================

BASE_DIR = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"
INPUT_FILE  = os.path.join(BASE_DIR, "input", "speech_classification_cleaned_dataset.xlsx")
OUT_JSONL = os.path.join(BASE_DIR, "output", "batch_input_reasoning_yes.jsonl")


PROMPT_ID = "pmpt_69d2cd341784819087482318fc9b963600c11e7edd71440b"

# =========================
# LOAD DATA
# =========================
df = pd.read_excel(INPUT_FILE)

df = df.dropna(subset=["paragraph_clean"])
df["party"] = df["party"].fillna("Unknown")

# =========================
# CREATE JSONL
# =========================
MAX_CHARS = 4000

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for i, row in df.iterrows():

        qid = str(row["q_id"])
        party = str(row["party"])
        passage = str(row["paragraph_clean"])[:MAX_CHARS]

        passage = passage.replace("\n", " ").strip()

        record = {
            "custom_id": qid,
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-4.1-2025-04-14",

                "prompt": {
                    "id": PROMPT_ID,
                    "version": "1",   # optional but safe
                    "variables": {
                        "reasoning_required": "YES",
                        "party": party,
                        "passage": passage
                    }
                },

                "input": " ",

                "temperature": 0,
                "max_output_tokens": 300,

                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "dw_score",
                        "schema": {
                            "type": "object",
                            "properties": {
                                "score": {
                                    "type": "integer",
                                    "minimum": 1,
                                    "maximum": 5
                                },
                                "reason": {
                                    "anyOf": [
                                        {"type": "string"},
                                        {"type": "null"}
                                    ]
                                }
                            },
                            "required": ["score", "reason"],
                            "additionalProperties": False
                        },
                        "strict": True
                    }
                },

                "metadata": {
                    "task": "dw_reasoning_yes"
                }
            }
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ JSONL file created:", OUT_JSONL)



✅ JSONL file created: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/batch_input_reasoning_yes.jsonl


In [22]:

from openai import OpenAI
import os


OPENAI_KEY_PATH = os.path.join(BASE_DIR, "input", "api's", "openai_api.txt")

# =========================
# Load API key + client
# =========================
with open(OPENAI_KEY_PATH, "r", encoding="utf-8") as f:
    OPENAI_API_KEY = f.read().strip()
if not OPENAI_API_KEY:
    raise RuntimeError("openai_api.txt is empty")

client = OpenAI(api_key=OPENAI_API_KEY)

with open(OUT_JSONL, "rb") as batch_file:
    file = client.files.create(
        file=batch_file,
        purpose="batch"
    )

print("File ID:", file.id)

File ID: file-JNCm9QgBEmDaFhFBxgVdXh


In [23]:
batch = client.batches.create(
    input_file_id=file.id,
    endpoint="/v1/responses",
    completion_window="24h"
)

print("Batch ID:", batch.id)
print("Status:", batch.status)

Batch ID: batch_69d2de202f508190bf1769b0066c20d4
Status: validating


### score only 

In [21]:
import json
import os
import pandas as pd

# =========================
# CONFIG
# =========================

BASE_DIR = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"
INPUT_FILE  = os.path.join(BASE_DIR, "input", "speech_classification_cleaned_dataset.xlsx")
OUT_JSONL   = os.path.join(BASE_DIR, "output", "batch_input_score_only.jsonl")

PROMPT_ID = "pmpt_69d2d58a00408190bd04c4201ad61d7204a8863f03986b3b"

# =========================
# LOAD DATA
# =========================
df = pd.read_excel(INPUT_FILE)

df = df.dropna(subset=["paragraph_clean"])
df["party"] = df["party"].fillna("Unknown")

# =========================
# CREATE JSONL
# =========================
MAX_CHARS = 4000

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for i, row in df.iterrows():

        qid = str(row["q_id"])
        party = str(row["party"])
        passage = str(row["paragraph_clean"])[:MAX_CHARS]

        passage = passage.replace("\n", " ").strip()

        record = {
            "custom_id": qid,
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-4.1-2025-04-14",
                "prompt": {
                    "id": PROMPT_ID,
                    "version": "3",
                    "variables": {          # ← this is what fills {party} and {passage}
                        "party": party,
                        "passage": passage
                    }
                },
                "input": " ",
                
                "temperature": 0,
                "max_output_tokens": 50,
                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "dw_score",
                        "schema": {
                            "type": "object",
                            "properties": {
                                "score": {"type": "integer", "minimum": 1, "maximum": 5}
                            },
                            "required": ["score"],
                            "additionalProperties": False
                        },
                        "strict": True
                    }
                },
                "metadata": {"task": "dw_score_only"}
            }
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ JSONL file created:", OUT_JSONL)

✅ JSONL file created: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/batch_input_score_only.jsonl


In [6]:
from decimal import Decimal
from pathlib import Path
import glob
import json
import os

FILES = [
    "output/batch_results/batch_input_reasoning_yes.jsonl",
    "output/batch_results/batch_input_score_only.jsonl",
]

# GPT-4.1 Batch pricing used for these jobs.
INPUT_COST_PER_M = Decimal("1.0")
OUTPUT_COST_PER_M = Decimal("4.0")


def _read_first_record(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                return json.loads(line)
    raise ValueError(f"{path} is empty")


def resolve_completed_batch_output(path: str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Batch file not found: {path}")

    first_record = _read_first_record(path)
    if first_record.get("response", {}).get("body", {}).get("usage"):
        return path

    body = first_record.get("body") or {}
    metadata = body.get("metadata") or {}
    task = metadata.get("task")
    if not task:
        raise ValueError(
            f"{path} is missing both response.body.usage and body.metadata.task, "
            "so the completed batch output file cannot be located."
        )

    candidates = sorted(glob.glob("temp/batch_*_output.jsonl"))
    matches = []
    for candidate in candidates:
        candidate_first = _read_first_record(candidate)
        candidate_body = candidate_first.get("response", {}).get("body", {})
        candidate_task = (candidate_body.get("metadata") or {}).get("task")
        candidate_usage = candidate_body.get("usage")
        if candidate_task == task and candidate_usage:
            matches.append(candidate)

    if len(matches) != 1:
        raise ValueError(
            f"{path} is a batch input JSONL for task '{task}', but I found {len(matches)} "
            "matching completed outputs in temp/. Put the completed output JSONL in "
            "output/batch_results/ or make the match unambiguous."
        )

    return matches[0]


def summarize_batch_output(path: str) -> dict:
    completed_path = resolve_completed_batch_output(path)

    request_count = 0
    input_tokens = 0
    output_tokens = 0
    task = None

    with open(completed_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue

            record = json.loads(line)
            body = record.get("response", {}).get("body", {})
            usage = body.get("usage")
            if not usage:
                raise ValueError(
                    f"{completed_path}:{line_no} is missing response.body.usage. "
                    "Costs can only be computed from completed batch output JSONLs."
                )

            request_count += 1
            input_tokens += usage["input_tokens"]
            output_tokens += usage["output_tokens"]

            metadata = body.get("metadata") or {}
            if task is None:
                task = metadata.get("task")

    input_cost = (Decimal(input_tokens) / Decimal(1_000_000)) * INPUT_COST_PER_M
    output_cost = (Decimal(output_tokens) / Decimal(1_000_000)) * OUTPUT_COST_PER_M

    return {
        "input_path": path,
        "completed_path": completed_path,
        "task": task,
        "request_count": request_count,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost,
    }


grand_requests = 0
grand_input_tokens = 0
grand_output_tokens = 0
grand_input_cost = Decimal("0")
grand_output_cost = Decimal("0")

for path in FILES:
    summary = summarize_batch_output(path)
    grand_requests += summary["request_count"]
    grand_input_tokens += summary["input_tokens"]
    grand_output_tokens += summary["output_tokens"]
    grand_input_cost += summary["input_cost"]
    grand_output_cost += summary["output_cost"]

    print(f"Input file: {summary['input_path']}")
    print(f"Completed output file: {summary['completed_path']}")
    print(f"  Task: {summary['task']}")
    print(f"  Requests: {summary['request_count']}")
    print(f"  Input tokens: {summary['input_tokens']:,}")
    print(f"  Output tokens: {summary['output_tokens']:,}")
    print(f"  Input cost (USD): ${summary['input_cost']:.6f}")
    print(f"  Output cost (USD): ${summary['output_cost']:.6f}")
    print(f"  Total cost (USD): ${summary['total_cost']:.6f}")
    print()

grand_total_cost = grand_input_cost + grand_output_cost

print("Combined total")
print(f"  Requests: {grand_requests}")
print(f"  Input tokens: {grand_input_tokens:,}")
print(f"  Output tokens: {grand_output_tokens:,}")
print(f"  Input cost (USD): ${grand_input_cost:.6f}")
print(f"  Output cost (USD): ${grand_output_cost:.6f}")
print(f"  Total cost (USD): ${grand_total_cost:.6f}")


Input file: output/batch_results/batch_input_reasoning_yes.jsonl
Completed output file: temp/batch_69d2d187c0f0819081dee9a228a27d14_output.jsonl
  Task: dw_reasoning_yes
  Requests: 511
  Input tokens: 363,937
  Output tokens: 64,178
  Input cost (USD): $0.363937
  Output cost (USD): $0.256712
  Total cost (USD): $0.620649

Input file: output/batch_results/batch_input_score_only.jsonl
Completed output file: temp/batch_69d2de202f508190bf1769b0066c20d4_output.jsonl
  Task: dw_score_only
  Requests: 511
  Input tokens: 287,798
  Output tokens: 4,599
  Input cost (USD): $0.287798
  Output cost (USD): $0.018396
  Total cost (USD): $0.306194

Combined total
  Requests: 1022
  Input tokens: 651,735
  Output tokens: 68,777
  Input cost (USD): $0.651735
  Output cost (USD): $0.275108
  Total cost (USD): $0.926843
